In [1]:
import pandas as pd
from pathlib import Path

from pydantic import BaseModel

import os
from openai import OpenAI
from dotenv import load_dotenv  
from pprint import pprint

from tqdm import tqdm
import constants

from prompting_utils import create_system_prompt
from model_utils import from_series_to_basemodel

import json
from dotenv import load_dotenv


from pathlib import Path

# Preliminari

In [2]:
# Configurazione OpenAI
load_dotenv()
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Parametri
base_dir = Path.cwd().parent
SYSTEM_PROMPT_FILE_NAME = constants.SYSTEM_PROMPT_6
TEMPERATURE = 0.0

MODEL = constants.TUNED_GPT_4_1_NANO_OVERSAMPLE
RESULTS_FILE_NAME = 'new_results_' + 'gpt-4.1-nano_FT_OS' + '.jsonl'

PYDANTIC_MODEL = constants.RectalCancerStagingData

#Carica system prompt da file
system_prompt_path = base_dir / "data" / "prompts" / SYSTEM_PROMPT_FILE_NAME
system_prompt = create_system_prompt(system_prompt_path, PYDANTIC_MODEL)

In [3]:
if True:
    print(system_prompt)

<ruolo>
Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite RM.
Il tuo compito è estrarre informazioni strutturate dal referto RM fornito e restituire esclusivamente un oggetto JSON valido conforme allo schema seguente.
</ruolo>

<schema>
{
  "morfologia": "solido_polipoide | solido_anulare | mucinoso",
  "ore_inizio": "int | null",
  "ore_fine": "int | null",
  "spessore_parietale": "int | null",
  "estensione_cranio_caudale": "int | null",
  "distanza_oai": "int | null",
  "posizione": {
    "basso": "no | si",
    "medio": "no | si",
    "alto": "no | si",
    "giunzione": "no | si"
  },
  "riflessione_peritoneale_anteriore": "sotto | cavallo | non_descritto",
  "infiltrazione_tessuto_adiposo": "no | si_5mm | si_5mm_plus",
  "infiltrazione_sfinteri": "no | si",
  "infiltrazione_organi_extra": "no | si",
  "infiltrazione_organi_dettagli": {
    "muscolo_elevatore": "no | si",
    "pavimento_pelvico": "no | si",
    "altro": "no | si"
  },
  "c

# Load Data

In [4]:
# Carichiamo i nostri file csv
file_names = {
    'train': constants.TRAIN_SPLIT_FILE_NAME,
    'validation': constants.VALIDATION_SPLIT_FILE_NAME,
    'test': constants.TEST_SPLIT_FILE_NAME,
}

paths = {split: Path('../data/' + file_name) for split, file_name in file_names.items()}

data = dict()
for split, path in paths.items():
    data[split] = pd.read_csv(path)

validation_data, test_data = data['validation'], data['test']
train_data = data['train']

################################
# Convert float columns to Int64
################################
float_cols = test_data.select_dtypes("float").columns
for col in float_cols:
    test_data[col] = test_data[col].round().astype("Int64")
    validation_data[col] = validation_data[col].round().astype("Int64")
    train_data[col] = train_data[col].round().astype("Int64")
    
# Check duplicatest
assert set(test_data.id) & set(validation_data.id) & set(train_data.id)== set(), "There are overlapping IDs between test and validation sets!"

print(f"{len(train_data) = }")
print(f"{len(test_data) = }")
print(f"{len(validation_data) = }")

len(train_data) = 187
len(test_data) = 65
len(validation_data) = 63


# Generazione

## Preliminari generazione

In [5]:
MODEL

'ft:gpt-4.1-nano-2025-04-14:luca-tramonti:report-extractor-oversampling:DPqaFyZI:ckpt-step-486'

In [6]:
# Funzione generatrice
def extract_data_from_report(model: str,
                             report_text: str,
                             system_prompt: str = system_prompt,
                             temperature: float = TEMPERATURE,
                             output_format: type[BaseModel] = PYDANTIC_MODEL):
    response = client.responses.parse(
        model=model,
        temperature=temperature,
        input=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': report_text},
        ],
        text_format=output_format,
        #reasoning={
        #    "effort": "high"
        #},
        #text={
        #    "verbosity": "low"
        #}
    )
    return response

In [7]:
# Esempio
if True:
    result = extract_data_from_report(MODEL, data['test'].iloc[0]['report_text'])

In [8]:
if False: # To see the full output
    pprint(result.model_dump())
if True:  # To see the string output text
    print(result.output_text)
if False:  # To see the parsed output as a pydantic BaseModel
    print(result.output_parsed)
if False:
    print(result.output_parsed.model_dump(mode='json'))

{"morfologia":"solido_polipoide","ore_inizio":null,"ore_fine":null,"spessore_parietale":null,"estensione_cranio_caudale":45,"distanza_oai":70,"posizione":{"basso":"no","medio":"si","alto":"no","giunzione":"no"},"riflessione_peritoneale_anteriore":"non_descritto","infiltrazione_tessuto_adiposo":"si_5mm","infiltrazione_sfinteri":"no","infiltrazione_organi_extra":"no","infiltrazione_organi_dettagli":{"muscolo_elevatore":"no","pavimento_pelvico":"no","altro":"no"},"coinvolgimento_riflessione_peritoneale":"no","coinvolgimento_fascia_mesorettale":"no","numero_linfonodi_non_conosciuto":"conosciuto","linfonodi_sospetti":0,"sedi_linfonodi":{"mesorettali":"si","rettali_superiori":"no","otturatori":"si","iliaci":"no","altro":"no"},"depositi_tumorali":"no","emvi":"no","stadio_T":"T3ab","stadio_N":"N+","mrf":"no","metastasi":"MX"}


## Inference

In [9]:
print(MODEL)
df = test_data
ids_test = []
actual_test = []
predicted_test = []
splits_test = []
for i in tqdm(range(df.shape[0])):
    row = df.iloc[i]
    splits_test.append(row['split'])
    output = extract_data_from_report(model=MODEL, report_text=row[constants.REPORT_COLUMN_NAME])
    real = from_series_to_basemodel(row, PYDANTIC_MODEL)
    ids_test.append(row[constants.ID_COMULM_NAME])
    actual_test.append(real.model_dump(mode='json'))
    if output is None:
        predicted_test.append('no output')
    else:
        predicted_test.append(output.output_parsed.model_dump(mode='json'))

ft:gpt-4.1-nano-2025-04-14:luca-tramonti:report-extractor-oversampling:DPqaFyZI:ckpt-step-486


100%|██████████| 65/65 [03:10<00:00,  2.93s/it]


In [10]:
print(MODEL)
df = validation_data
ids_validation = []
actual_validation = []
predicted_validation = []
splits_validation = []
for i in tqdm(range(df.shape[0])):
    row = df.iloc[i]
    splits_validation.append(row['split'])
    output = extract_data_from_report(model=MODEL, report_text=row[constants.REPORT_COLUMN_NAME])
    real = from_series_to_basemodel(row, PYDANTIC_MODEL)
    ids_validation.append(row[constants.ID_COMULM_NAME])
    actual_validation.append(real.model_dump(mode='json'))
    if output is None:
        predicted_validation.append('no output')
    else:
        predicted_validation.append(output.output_parsed.model_dump(mode='json'))

ft:gpt-4.1-nano-2025-04-14:luca-tramonti:report-extractor-oversampling:DPqaFyZI:ckpt-step-486


100%|██████████| 63/63 [02:16<00:00,  2.16s/it]


In [11]:
print(MODEL)
df = train_data
ids_train = []
actual_train = []
predicted_train = []
splits_train = []
for i in tqdm(range(df.shape[0])):
    row = df.iloc[i]
    splits_train.append(row['split'])
    output = extract_data_from_report(model=MODEL, report_text=row[constants.REPORT_COLUMN_NAME])
    real = from_series_to_basemodel(row, PYDANTIC_MODEL)
    ids_train.append(row[constants.ID_COMULM_NAME])
    actual_train.append(real.model_dump(mode='json'))
    if output is None:
        predicted_train.append('no output')
    else:
        predicted_train.append(output.output_parsed.model_dump(mode='json'))

ft:gpt-4.1-nano-2025-04-14:luca-tramonti:report-extractor:DNTiYe0j


100%|██████████| 187/187 [07:13<00:00,  2.32s/it]


In [11]:
ids = ids_validation + ids_test
actual = actual_validation + actual_test
predicted = predicted_validation + predicted_test
splits = splits_validation + splits_test

In [12]:
results_dicts = []
assert len(ids) == len(actual) == len(predicted) == len(splits)
for id, act, pred, split in zip(ids, actual, predicted, splits):
    results_dicts.append(
        {
            'model': MODEL,
            'split': split,
            'id': int(id),
            'actual': act,
            'prediction': pred
        }
    )
# Salvataggio
with open(base_dir / 'data' / 'inference' / RESULTS_FILE_NAME, 'w', encoding='utf-8') as f:
    for r in results_dicts:
        f.write(json.dumps(r) + '\n')